In [6]:
import cv2, numpy as np, math, os, csv, re
from itertools import combinations
import matplotlib.pyplot as plt

IMAGE_FOLDER = '/content/drive/MyDrive/ASLDATASET'
OUT_CSV = '/content/drive/MyDrive/ASL_results_debug.csv'
WRIST_REMOVE_RATIO = 0.14


TH = {
    'min_area': 2000,
    'min_extent': 0.18,
    'circ_B': 0.38,
    'ar_PQ': 1.20,
    's_PQ': 0.70,
    'ar_V': 0.62,
    's_V': 0.70,
    's_N': 0.88,
    'up_D': 0.42,
    'circ_D': 0.42,
    's_D': 0.88,
    's_S': 0.90,
    'circ_S': 0.50,
    'circ_M': 0.46,
    's_M': 0.86,
    'circ_O': 0.55,
    'ar_O_diff': 0.28,
    'circ_C': 0.45,
    's_C': 0.82,
    'xspread_U': 0.18,
    'defect_depth_rel': 0.03,
    'defect_depth_px_min': 10,
    'defect_angle_max': 90.0,
}

def safe_acos(x):
    return math.acos(max(-1.0, min(1.0, x)))
def dist(a,b): return math.hypot(a[0]-b[0], a[1]-b[1])

#feature extractor
def extract_hand_features_fixed(frame):
    roi = frame.copy()
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    ycrcb = cv2.cvtColor(roi, cv2.COLOR_BGR2YCrCb)

    mask_hsv = cv2.inRange(hsv, np.array([0,10,60]), np.array([20,150,255]))
    mask_ycrcb = cv2.inRange(ycrcb, np.array([0,133,77]), np.array([255,173,127]))
    mask = cv2.bitwise_and(mask_hsv, mask_ycrcb)

    kernel = np.ones((3,3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.GaussianBlur(mask, (5,5), 0)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, None, None
    cnt = max(contours, key=lambda c: cv2.contourArea(c))
    area = float(cv2.contourArea(cnt))
    if area <= 0:
        return None, None, None

    x,y,w,h = cv2.boundingRect(cnt)
    perimeter = float(cv2.arcLength(cnt, True))
    hull = cv2.convexHull(cnt)
    hull_area = cv2.contourArea(hull) if len(hull)>0 else 1.0
    solidity = float(area / hull_area) if hull_area>1e-6 else 0.0
    circularity = float((4.0 * math.pi * area) / (perimeter**2)) if perimeter>1e-6 else 0.0
    aspect_ratio = float(w)/h if h>0 else 0.0
    extent = float(area)/(w*h) if (w*h)>0 else 0.0
    M = cv2.moments(cnt)
    cx = int(M['m10']/M['m00']) if M['m00']!=0 else x+w//2
    cy = int(M['m01']/M['m00']) if M['m00']!=0 else y+h//2
    top_y = int(np.min(cnt[:,0,1]))

    # wrist removal
    mask2 = mask.copy()
    wrist_cut = y + int(h * (1.0 - WRIST_REMOVE_RATIO))
    wrist_cut = min(max(wrist_cut, y), y+h-1)
    mask2[wrist_cut : y+h, :] = 0

    contours2, _ = cv2.findContours(mask2, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours2:
        cnt2 = max(contours2, key=lambda c: cv2.contourArea(c))
        if cv2.contourArea(cnt2) > 0.25*area:
            cnt_used = cnt2
        else:
            cnt_used = cnt
    else:
        cnt_used = cnt

    hull_used = cv2.convexHull(cnt_used)
    hull_area_used = cv2.contourArea(hull_used) if len(hull_used)>0 else hull_area
    perimeter_used = float(cv2.arcLength(cnt_used, True))
    M2 = cv2.moments(cnt_used)
    cx2 = int(M2['m10']/M2['m00']) if M2['m00']!=0 else cx
    cy2 = int(M2['m01']/M2['m00']) if M2['m00']!=0 else cy

    # convexity defects
    fingertip_count = 0
    defect_depths = []
    hull_indices = cv2.convexHull(cnt_used, returnPoints=False)
    if hull_indices is not None and len(hull_indices) > 3:
        defects = cv2.convexityDefects(cnt_used, hull_indices)
        if defects is not None:
            for i in range(defects.shape[0]):
                s,e,f,d = defects[i,0]
                start = tuple(cnt_used[s][0]); end = tuple(cnt_used[e][0]); far = tuple(cnt_used[f][0])
                a = dist(start,end); b = dist(start,far); c = dist(end,far)
                if b>1e-6 and c>1e-6:
                    val = (b*b + c*c - a*a) / (2*b*c)
                    angle = math.degrees(safe_acos(val))
                    depth_px = d / 256.0 * perimeter_used if isinstance(d,(int,float)) else float(d)
                    cond_depth_rel = (d / perimeter_used) if perimeter_used>0 else 0.0
                    if angle <= TH['defect_angle_max'] and cond_depth_rel >= TH['defect_depth_rel'] and d >= TH['defect_depth_px_min']:
                        if far[1] < (cy2 + 0.15*h):
                            fingertip_count += 1
                            defect_depths.append(cond_depth_rel)
    fingertip_count = min(max(0, fingertip_count) + 1, 5)



    hull_pts = hull_used.reshape(-1,2) if len(hull_used)>0 else np.array([[cx2,cy2]])
    xs = hull_pts[:,0].tolist(); ys = hull_pts[:,1].tolist()
    diag = math.hypot(w,h) if (w>0 and h>0) else 1.0
    hull_dists = [math.hypot(int(px)-cx2, int(py)-cy2) for (px,py) in hull_pts]
    hull_spread = max(hull_dists)/diag if hull_dists else 0.0
    xspread = (max(xs)-min(xs))/diag if xs else 0.0
    yspread = (max(ys)-min(ys))/diag if ys else 0.0

    ft_pts = []
    if defect_depths:
        approx = cnt_used.squeeze()
        yvals = approx[:,1]
        idxs = np.argsort(yvals)[:5]
        ft_pts = [tuple(approx[int(i)]) for i in idxs]
    else:
        ft_pts = [(min(xs), ys[xs.index(min(xs))]), (max(xs), ys[xs.index(max(xs))])]

    ft_spread = 0.0
    if len(ft_pts) >= 2:
        maxd = 0.0
        for a,b in combinations(ft_pts,2):
            maxd = max(maxd, dist(a,b))
        ft_spread = maxd/diag

    upness = float((cy2 - top_y) / (h if h>0 else 1.0))
    avg_def = float(np.mean(defect_depths)) if len(defect_depths)>0 else 0.0

    features = {
        'area': area, 'perimeter': perimeter, 'solidity': round(solidity,4),
        'circularity': round(circularity,4), 'aspect_ratio': round(aspect_ratio,4),
        'extent': round(extent,4), 'palm_center': (cx2,cy2), 'top_y': top_y,
        'bounding_box': (x,y,w,h), 'finger_count': int(fingertip_count),
        'defect_depths': defect_depths, 'avg_defect_depth': round(avg_def,4),
        'hull_spread': round(hull_spread,4),
        'xspread': round(xspread,4), 'yspread': round(yspread,4),
        'fingertip_spread': round(ft_spread,4), 'upness': round(upness,4),
        'mask': mask2, 'contour': cnt_used, 'roi': roi
    }
    return features, roi, mask2

def pred_contains_letter(pred_str, true_letter):
    toks = re.findall(r"[A-Z]", pred_str.upper())
    if true_letter in toks:
        return True
    toks_multi = re.findall(r"[A-Z]+", pred_str.upper())
    return any(true_letter == t or true_letter in t for t in toks_multi)

#logic
def classify_rules_safe(f):

    if f is None:
        return "No-hand"
    area = f['area']; ext = f['extent']
    s = f['solidity']; circ = f['circularity']; ar = f['aspect_ratio']
    fingers = f['finger_count']; upness = f['upness']; xspread = f['xspread']; ft_spread = f['fingertip_spread']
    hull_spread = f.get('hull_spread', 0.0)


    if area < TH['min_area'] or ext < TH['min_extent']:
        return "No-hand/Poor-capture"

    if (
        (fingers == 2 and s >= 0.90 and circ >= 0.50)  # classic two-finger T (keeps old behavior)
        or
        (fingers == 1 and s >= 0.89 and ft_spread >= 0.60 and circ >= 0.48 and 0.60 <= ar <= 0.80)
    ):
        return "T"

    if fingers <= 1 and s >= TH['s_S'] and circ >= TH['circ_S'] and ext >= 0.55:
        return "S"

    if fingers <= 1 and s >= TH['s_M'] and circ >= TH['circ_M'] and 0.48 <= ar <= 0.95:
        return "M"


    if fingers == 1 and ar > 1.25 and ft_spread > 0.80 and xspread > 0.75:
        return "G"

    if fingers == 3 and ar > 1.10 and xspread > 0.70:
        return "H"

    if fingers == 2 and ft_spread < 0.05 and xspread > 0.55 and 0.36 <= circ <= 0.40 and ext < 0.56:
        return "C"
    if fingers == 2 and 0.67 <= ar <= 0.85 and 0.70 <= s <= 0.82 and 0.012 <= ft_spread <= 0.045 and xspread >= 0.45:
        if circ < 0.42:
            return "X"

    if (
        ((0.40 <= circ <= 0.52) and (abs(ar - 1.0) <= 0.18) and ft_spread < 0.08 and (0.44 <= ext <= 0.56))
        or

        ((circ < 0.40) and (abs(ar - 1.0) <= 0.15) and xspread >= 0.65 and ft_spread < 0.05 and 0.48 <= ext <= 0.54)
    ):
        return "O"



    if fingers == 2 and ar >= TH['ar_PQ'] and s < TH['s_PQ']:
        return "P"
    if fingers == 2 and ar > 1.15 and s < 0.76:
        return "Q"

    if fingers == 2 and s < 0.62 and xspread > 0.55 and ar > 0.68:
        return "L"
    if fingers == 1 and ar > 1.05 and upness > 0.30:
        return "L"

    if fingers == 1 and ft_spread >= 0.68 and 0.80 <= s <= 0.92 and 0.45 <= ar <= 0.72:
        return "E"

    if fingers == 1 and ft_spread >= 0.60 and ar > 0.80 and s < 0.78:
        return "Y"

    if fingers == 1 and s >= 0.86 and 0.40 <= ft_spread <= 0.60:
        return "U"



    if fingers == 1 and 0.35 <= ft_spread <= 0.52 and 0.80 <= s <= 0.84 and ar < 0.65 and upness > 0.48:
        return "R"

    if fingers == 1 and upness > TH['up_D'] and circ < TH['circ_D'] and s < TH['s_D']:
        return "D"


    if fingers == 2 and 0.44 <= ar <= 0.60 and 0.68 <= s <= 0.76 and ft_spread < 0.05:
        return "K"


    if fingers == 2 and 0.72 <= ar <= 0.90 and 0.70 <= s <= 0.80 and 0.02 <= ft_spread <= 0.04:
        return "X"

    if fingers == 2 and s >= 0.85 and ft_spread < 0.035 and ar < 0.55:
        return "I"

    if fingers == 2 and s >= TH['s_N'] and 0.54 <= ar <= 0.60 and ft_spread < 0.04:
        return "N"

    if fingers == 2 and xspread >= 0.35 and ft_spread <= 0.06 and ar < TH['ar_V'] and s > TH['s_V']:
        return "V"

    if fingers >= 4 and circ < 0.23 and s < 0.82:
        return "F"


    if fingers >= 4:
        return "B"


    if ft_spread >= 0.45 and 0.38 <= circ <= 0.525 and s >= 0.82 and 0.45 <= ar <= 1.2:
        return "C"


    if fingers == 2 and ft_spread < 0.04 and s > 0.82:
        if circ >= TH['circ_S'] and s >= TH['s_S']:
            return "S"

        return "A"

    return "W"






def run_and_eval_sample_accuracy_print_features(folder, out_csv_path, show_vis=False):

    all_files = sorted([f for f in os.listdir(folder) if f.lower().endswith(('.jpg','.png','.jpeg'))])


    valid_files = []
    for f in all_files:
        m = re.search(r"([A-Za-z])", f)
        if not m:
            continue
        base = m.group(1).upper()
        if base in ('J','Z'):
            continue
        valid_files.append(f)

    total = len(valid_files)
    if total == 0:
        print("No valid images found in", folder)
        return

    rows = []
    correct_samples = 0

    for idx, fname in enumerate(valid_files, start=1):
        path = os.path.join(folder, fname)
        img = cv2.imread(path)
        if img is None:
            print(f"[{idx}/{total}] Can't open {fname}"); continue
        f, roi, mask = extract_hand_features_fixed(img)
        pred = classify_rules_safe(f)

        m = re.search(r"([A-Za-z])", fname)
        true_letter = m.group(1).upper() if m else None

        correct = False
        if true_letter:
            correct = pred_contains_letter(pred, true_letter)
        if correct:
            correct_samples += 1


        rows.append({
            'image': fname,
            'true_letter': true_letter,
            'prediction': pred,
            'correct': int(bool(correct)),
            'finger_count': f['finger_count'] if f else None,
            'solidity': f['solidity'] if f else None,
            'circularity': f['circularity'] if f else None,
            'aspect_ratio': f['aspect_ratio'] if f else None,
            'extent': f['extent'] if f else None,
            'hull_spread': f['hull_spread'] if f else None,
            'xspread': f['xspread'] if f else None,
            'fingertip_spread': f['fingertip_spread'] if f else None,
            'upness': f['upness'] if f else None
        })




    os.makedirs(os.path.dirname(out_csv_path), exist_ok=True)
    with open(out_csv_path, 'w', newline='') as cf:
        fieldnames = ['image','true_letter','prediction','correct','finger_count','solidity','circularity','aspect_ratio','extent','hull_spread','xspread','fingertip_spread','upness']
        writer = csv.DictWriter(cf, fieldnames=fieldnames)
        writer.writeheader()
        for r in rows:
            writer.writerow(r)

    processed = len(rows)
    correct_n = correct_samples
    acc = (correct_n / processed) * 100.0 if processed>0 else 0.0


    print(f"Number of processed images: {processed}")
    print(f"Correct predictions: {correct_n}")
    print(f"Accuracy: {correct_n} / {processed}  ({acc:.2f}%)")

    return rows

results = run_and_eval_sample_accuracy_print_features(IMAGE_FOLDER, OUT_CSV)

Number of processed images: 83
Correct predictions: 65
Accuracy: 65 / 83  (78.31%)
